## Setup

In [8]:
import logging
from pathlib import Path
import re
import subprocess

import pandas as pd

logger = logging.getLogger

# Custom words to remove
EXCLUDE_LIST = ['[FREE DL]', '[DUPLOC10YRS]', '[Digital Bonus]', '[Premiere]', '[Bassrush Premiere]', '(DDD118)']
REPLACE_DICT = {
    "¿": "?"
}


def get_file_df(dir_path, glob_pat="*"):
    path = []
    parent = []
    name = []
    stem = []
    suffix = []
    for pp in dir_path.glob(glob_pat):
        path.append(pp.resolve())
        parent.append(pp.parent)
        name.append(pp.name)
        stem.append(pp.stem)
        suffix.append(pp.suffix)
    df = pd.DataFrame({"path": path, "parent": parent, "name": name, "stem": stem, "suffix": suffix})
    return df


def strip_digit_prefix(stem):
    # "2.Sub Basics - Shipment Pt.2" => "Sub Basics - Shipment Pt.2"
    return re.sub(r'^\d+\.', '', stem)


def strip_underscore_digits(stem):
    # "Criso & Breathworkz - Wicked_2051092604 - Criso" => "Criso & Breathworkz - Wicked - Criso"
    return re.sub(r'_\d+', '', stem)


def remove_words(series):
    replace = REPLACE_DICT
    remove = {s: "" for s in EXCLUDE_LIST}
    replace.update(remove)
    series = series.str.replace(replace)
    # Remove consecutive whitespace and strip
    series = series.str.replace(r'\s+', ' ', regex=True).str.strip()
    return series


def run_ffmpeg(row):
    assert row.path
    assert row.new_sample_fmt
    assert row.new_sample_rate
    assert row.new_path
    if row.new_path.is_file():
        logger.warning(f"File exists, skipping: {row.new_path}")
        return None
    else:
        command = [
            'ffmpeg',
            '-i', row.path,
            '-sample_fmt', row.new_sample_fmt,
            '-ar', row.new_sample_rate,
            row.new_path
        ]
        process = subprocess.Popen(command)
        return_code = process.wait()
        return int(return_code)


## Run

In [24]:
src_dir = Path("C:/Users/sesti/Music/Playlists/djutil/stage/materials")
dst_dir = Path("C:/Users/sesti/Music/Playlists/materials")

In [25]:
! ls $src_dir

0427 1 - LOW high.wav
0427 1 - LOW low.wav
0427 1 - metal.wav
0427 1 - synth.wav
0427 1 - Tk drums.wav
128 air bus.wav
128 arp.wav
128 backbeat.wav
128 bass 1.wav
128 bass 2.wav
128 beginning metal and bass.wav
128 dry bus.wav
128 hats 1.wav
128 hats 2.wav
128 kick 4floor.wav
128 metal hit.wav
calluna - drum noise 1.wav
fools return - drum bass sub.wav
fools return - drums 1.wav
fools return - drums 2.wav
fools return - sub.wav
fools return - synth 1.wav
fools return - synth 2.wav
jammy 4 edit.wav
spiral drill - aux bass.wav
spiral drill - D#.wav
spiral drill - drums.wav
spiral drill - F#.wav
spiral drill - hat shaker.wav
spiral drill - sub.wav
spiral drill - synth 1.wav
spiral drill - throb.wav
spiral drill - vital rhythm.wav
vital noise.wav


In [26]:
! ls $dst_dir

In [27]:
df = get_file_df(src_dir)
df["new_stem"] = df.stem.map(strip_digit_prefix).map(strip_underscore_digits)
df["new_stem"] = remove_words(df.new_stem).str.strip()
df["new_suffix"] = ".wav"
df["new_parent"] = dst_dir
df["new_sample_rate"] = str(44100)
df["new_sample_fmt"] = "s16"
df["new_path"] = df.apply(lambda row: row.new_parent / (row.new_stem + row.new_suffix), axis=1).values

df[['new_stem', 'new_path']]

,new_stem,new_path
0,0427 1 - LOW high,C:\Users\sesti\Music\Playlists\materials\0427 ...
1,0427 1 - LOW low,C:\Users\sesti\Music\Playlists\materials\0427 ...
2,0427 1 - metal,C:\Users\sesti\Music\Playlists\materials\0427 ...
3,0427 1 - synth,C:\Users\sesti\Music\Playlists\materials\0427 ...
4,0427 1 - Tk drums,C:\Users\sesti\Music\Playlists\materials\0427 ...
5,128 air bus,C:\Users\sesti\Music\Playlists\materials\128 a...
6,128 arp,C:\Users\sesti\Music\Playlists\materials\128 a...
7,128 backbeat,C:\Users\sesti\Music\Playlists\materials\128 b...
8,128 bass 1,C:\Users\sesti\Music\Playlists\materials\128 b...
9,128 bass 2,C:\Users\sesti\Music\Playlists\materials\128 b...


In [28]:
df.apply(run_ffmpeg, axis=1)

0     0
1     0
2     0
3     0
4     0
5     0
6     0
7     0
8     0
9     0
10    0
11    0
12    0
13    0
14    0
15    0
16    0
17    0
18    0
19    0
20    0
21    0
22    0
23    0
24    0
25    0
26    0
27    0
28    0
29    0
30    0
31    0
32    0
33    0
dtype: int64

In [29]:
! ls $dst_dir

0427 1 - LOW high.wav
0427 1 - LOW low.wav
0427 1 - metal.wav
0427 1 - synth.wav
0427 1 - Tk drums.wav
128 air bus.wav
128 arp.wav
128 backbeat.wav
128 bass 1.wav
128 bass 2.wav
128 beginning metal and bass.wav
128 dry bus.wav
128 hats 1.wav
128 hats 2.wav
128 kick 4floor.wav
128 metal hit.wav
calluna - drum noise 1.wav
fools return - drum bass sub.wav
fools return - drums 1.wav
fools return - drums 2.wav
fools return - sub.wav
fools return - synth 1.wav
fools return - synth 2.wav
jammy 4 edit.wav
spiral drill - aux bass.wav
spiral drill - D#.wav
spiral drill - drums.wav
spiral drill - F#.wav
spiral drill - hat shaker.wav
spiral drill - sub.wav
spiral drill - synth 1.wav
spiral drill - throb.wav
spiral drill - vital rhythm.wav
vital noise.wav


In [ ]:
def get_file_df(dir_path, glob_pat="*"):
    path = []
    parent = []
    name = []
    stem = []
    suffix = []
    for pp in dir_path.glob(glob_pat):
        path.append(pp.resolve())
        parent.append(pp.parent)
        name.append(pp.name)
        stem.append(pp.stem)
        suffix.append(pp.suffix)
    df = pd.DataFrame({"path": path, "parent": parent, "name": name, "stem": stem, "suffix": suffix})
    return df


def strip_digit_prefix(stem):
    # "2.Sub Basics - Shipment Pt.2" => "Sub Basics - Shipment Pt.2"
    return re.sub(r'^\d+\.', '', stem)


def strip_underscore_digits(stem):
    # "Criso & Breathworkz - Wicked_2051092604 - Criso" => "Criso & Breathworkz - Wicked - Criso"
    return re.sub(r'_\d+', '', stem)


def remove_words(series):
    replace = REPLACE_DICT
    remove = {s: "" for s in EXCLUDE_LIST}
    replace.update(remove)
    series = series.str.replace(replace)
    # Remove consecutive whitespace and strip
    series = series.str.replace(r'\s+', ' ', regex=True).str.strip()
    return series


def run_ffmpeg(row):
    assert row.path
    assert row.new_sample_fmt
    assert row.new_sample_rate
    assert row.new_path
    if row.new_path.is_file():
        logger.warning(f"File exists, skipping: {row.new_path}")
        return None
    else:
        command = [
            'ffmpeg',
            '-i', row.path,
            '-sample_fmt', row.new_sample_fmt,
            '-ar', row.new_sample_rate,
            row.new_path
        ]
        process = subprocess.Popen(command)
        return_code = process.wait()
        return int(return_code)


In [ ]:
def run_ffmpeg(row):
    assert row.path
    assert row.new_sample_fmt
    assert row.new_sample_rate
    assert row.new_path
    if row.new_path.is_file():
        logger.warning(f"File exists, skipping: {row.new_path}")
        return None
    else:
        command = [
            'ffmpeg',
            '-i', row.path,
            '-sample_fmt', row.new_sample_fmt,
            '-ar', row.new_sample_rate,
            row.new_path
        ]
        process = subprocess.Popen(command)
        return_code = process.wait()
        return int(return_code)
        


In [ ]:
def get_file_df(dir_path, glob_pat="*"):
    path = []
    parent = []
    name = []
    stem = []
    suffix = []
    for pp in dir_path.glob(glob_pat):
        path.append(pp.resolve())
        parent.append(pp.parent)
        name.append(pp.name)
        stem.append(pp.stem)
        suffix.append(pp.suffix)
    df = pd.DataFrame({"path": path, "parent": parent, "name": name, "stem": stem, "suffix": suffix})
    return df